In [1]:
import pandas as pd
import torch
import os
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from datasets import Dataset
from transformers import BertTokenizer, TrainingArguments, Trainer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [3]:
df = pd.read_csv("C:/Users/User/Documents/devanasokan_fyp/preparation/modeldata.csv")

In [4]:
print(df.columns)
print(df.shape)

Index(['verse_id', 'song_id', 'ori_track_name', 'clean_track_name',
       'all_artists', 'primary_artist', 'artist_genres', 'main_genre',
       'explicit', 'section', 'verse', 'language', 'language.1', 'confidence',
       'confidence.1', 'label'],
      dtype='str')
(22878, 16)


In [ ]:
# fahh

In [5]:
# Class balance check
print(df['label'].value_counts())

label
0    11439
1    11439
Name: count, dtype: int64


In [6]:
# Select only the columns we need
df = df[['verse', 'label']]

In [7]:
# Convert to Hugging Face format
dataset = Dataset.from_pandas(df)

In [8]:
# Fixes the [WinError 3] by explicitly setting a local cache directory
cache_dir = "C:/Users/User/Documents/devanasokan_fyp/huggingface_cache"
if not os.path.exists(cache_dir):
    os.makedirs(cache_dir)

In [9]:
# Tell Hugging Face to use this directory
os.environ['HF_HOME'] = cache_dir

# This turns off the annoying symlink warning
os.environ['HF_HUB_DISABLE_SYMLINKS_WARNING'] = '1'

In [ ]:
# Initiate tokenizer with the cache path
tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased', cache_dir=cache_dir)

In [11]:
def preprocess_function(examples):
    # This creates the 'input_ids' and 'attention_mask' BERT needs
    return tokenizer(examples["verse"], truncation=True, padding="max_length", max_length=512)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

Map: 100%|██████████| 22878/22878 [00:04<00:00, 4860.27 examples/s]


In [12]:
# 80% Train, 20% Test
full_dataset = tokenized_dataset.train_test_split(test_size=0.2, seed=42)

In [13]:
print(full_dataset) 
# If it shows {'train': ..., 'test': ...}, it is already split!

DatasetDict({
    train: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 18302
    })
    test: Dataset({
        features: ['verse', 'label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 4576
    })
})


In [ ]:
model_name = "distilbert-base-uncased" # Or any model from the Hugging Face Hub

# 1. Load the tokenizer (must match the model)
tokenizer = AutoTokenizer.from_pretrained(model_name)

# 2. Load the model with a classification head
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4147.73it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [15]:
def model_init():
    return AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)


In [16]:
import evaluate
metric = evaluate.load("accuracy")

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # argmax picks the highest probability (0 or 1)
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [18]:
# Set training arguments
training_args = TrainingArguments(
    output_dir="./results",          # Folder where checkpoints are saved
    eval_strategy="epoch",           # Run evaluation after every epoch
    save_strategy="epoch",           # Save model after every epoch
    learning_rate=2e-5,               # Default value; overridden in the final run
    per_device_train_batch_size=16,   # Default value; overridden in the final run
    per_device_eval_batch_size=16,    # Batch size for evaluation
    num_train_epochs=2,               # Default value; overridden in the final run
    weight_decay=0.01,                # Regularization to prevent overfitting
    load_best_model_at_end=True,      # Keeps the best version of the model
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()    # Use Mixed Precision if on GPU for 2x speed
)


In [19]:
trainer = Trainer(
    model_init=model_init,
    args=training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

print(trainer.compute_metrics)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6527.29it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<function compute_metrics at 0x00000249E0E9C7C0>


In [20]:
# Hyperparameter search space definition
def hp_space(trial):
    return {
        "learning_rate": trial.suggest_float("learning_rate", 1e-5, 5e-5, log=True),
        "per_device_train_batch_size": trial.suggest_categorical(
            "per_device_train_batch_size", [8, 16]
        ),
        "num_train_epochs": trial.suggest_int("num_train_epochs", 2, 4),
        "weight_decay": trial.suggest_float("weight_decay", 0.0, 0.1),
    }

# Run the hyperparameter search
best_run = trainer.hyperparameter_search(
    direction="maximize",
    backend="optuna",
    hp_space=hp_space,
    n_trials=6,
    compute_objective=lambda metrics: metrics["eval_accuracy"],
)

print("Best run: ", best_run)
print("Best hyperparameters found: ", best_run.hyperparameters)

[I 2026-06-11 12:43:11,127] A new study created in memory with name: no-name-53bd433f-ee02-4342-aba6-3090c57254f1
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6360.60it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.334378,0.311199,0.859484
2,0.209073,0.336522,0.878059
3,0.148196,0.424114,0.881337
4,0.098254,0.521871,0.885052


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.47it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-06-11 13:19:50,938] Trial 0 finished with value: 0.8850524475524476 and parameters: {'learning_rate': 1.7827933709057233e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.06856160592298861}. Best is trial 0 with value: 0.8850524475524476.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 6792.62it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight |

Epoch,Training Loss,Validation Loss,Accuracy
1,0.334393,0.312104,0.860358
2,0.200817,0.363343,0.876967
3,0.130491,0.452972,0.881337
4,0.074995,0.562941,0.884615


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.25s/it]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-06-11 13:43:14,437] Trial 1 finished with value: 0.8846153846153846 and parameters: {'learning_rate': 2.490221618410341e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.02469861648848878}. Best is trial 0 with value: 0.8850524475524476.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 8592.24it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | 

...

Epoch,Training Loss,Validation Loss,Accuracy
1,0.335984,0.322548,0.856643
2,0.184571,0.375581,0.878715
3,0.106096,0.510077,0.880900


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.09s/it]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
[I 2026-06-11 15:00:32,778] Trial 4 finished with value: 0.8809003496503497 and parameters: {'learning_rate': 3.959993921380352e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 3, 'weight_decay': 0.08960510904820053}. Best is trial 0 with value: 0.8850524475524476.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7228.69it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | 

Epoch,Training Loss,Validation Loss,Accuracy
1,0.333777,0.342898,0.863418


[I 2026-06-11 15:07:25,932] Trial 5 pruned. 


Best run:  BestRun(run_id='0', objective=0.8850524475524476, hyperparameters={'learning_rate': 1.7827933709057233e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.06856160592298861}, run_summary=None)
Best hyperparameters found:  {'learning_rate': 1.7827933709057233e-05, 'per_device_train_batch_size': 16, 'num_train_epochs': 4, 'weight_decay': 0.06856160592298861}


In [21]:
best_hyperparameters = best_run.hyperparameters
final_training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=best_hyperparameters["learning_rate"],
    per_device_train_batch_size=best_hyperparameters["per_device_train_batch_size"],
    per_device_eval_batch_size=16,
    num_train_epochs=best_hyperparameters["num_train_epochs"],
    weight_decay=best_hyperparameters["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available(),
)

final_trainer = Trainer(
    model=model_init(),
    args=final_training_args,
    train_dataset=full_dataset["train"],
    eval_dataset=full_dataset["test"],
    compute_metrics=compute_metrics,
)

final_trainer.train()

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 7366.05it/s]
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,0.332943,0.304739,0.865385
2,0.205460,0.332529,0.880682
3,0.149498,0.445140,0.880026
4,0.100079,0.530586,0.882212


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.35it/s]
There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].


TrainOutput(global_step=4576, training_loss=0.20228672527766728, metrics={'train_runtime': 2199.2202, 'train_samples_per_second': 33.288, 'train_steps_per_second': 2.081, 'total_flos': 9697673320808448.0, 'train_loss': 0.20228672527766728, 'epoch': 4.0})

In [22]:
history = pd.DataFrame(final_trainer.state.log_history)
print(history)


        loss  grad_norm  learning_rate     epoch  step  eval_loss  \
0   0.386497   7.964220   1.588774e-05  0.437063   500        NaN   
1   0.332943   6.497315   1.393976e-05  0.874126  1000        NaN   
2        NaN        NaN            NaN  1.000000  1144   0.304739   
3   0.256822   9.080848   1.199567e-05  1.311189  1500        NaN   
4   0.205460   9.891187   1.004769e-05  1.748252  2000        NaN   
5        NaN        NaN            NaN  2.000000  2288   0.332529   
6   0.180712   7.319865   8.099710e-06  2.185315  2500        NaN   
7   0.149498  15.411551   6.155624e-06  2.622378  3000        NaN   
8        NaN        NaN            NaN  3.000000  3432   0.445140   
9   0.144417   0.962546   4.207642e-06  3.059441  3500        NaN   
10  0.083012   0.040667   2.259659e-06  3.496503  4000        NaN   
11  0.100079  15.141699   3.116772e-07  3.933566  4500        NaN   
12       NaN        NaN            NaN  4.000000  4576   0.530586   
13       NaN        NaN           

In [ ]:
# Confusion matrix and classification report
from sklearn.metrics import classification_report, confusion_matrix

print("Classification Report:")
classification_report_output = classification_report(full_dataset["test"]["label"], np.argmax(trainer.predict(full_dataset["test"]).predictions, axis=-1))
print(classification_report_output)

print("Confusion Matrix:")
confusion_matrix_output = confusion_matrix(full_dataset["test"]["label"], np.argmax(trainer.predict(full_dataset["test"]).predictions, axis=-1))
print(confusion_matrix_output)

In [23]:
# Save the version currently in the final trainer's brain
final_trainer.save_model("./my_final_model")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.93it/s]


In [24]:
# TEST MODEL

from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

# Load the model and tokenizer from your local folder
path = "./my_final_model"
model = AutoModelForSequenceClassification.from_pretrained(path)
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased") # Ensure you saved the tokenizer there too!

# Create a 'pipeline' (the easiest way to use the model)
classifier = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 7364.39it/s]


In [25]:
# Test it on a new sentence
result = classifier("kiss it from my lips")
print(result)

[{'label': 'LABEL_1', 'score': 0.9721267223358154}]


In [34]:
import accelerate
print(accelerate.__version__)

1.13.0


In [35]:
import accelerate
import transformers
print(f"Accelerate version: {accelerate.__version__}")
print(f"Transformers version: {transformers.__version__}")

Accelerate version: 1.13.0
Transformers version: 5.3.0


In [36]:
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

Torch version: 2.7.1+cu118
CUDA available: True
CUDA version: 11.8
GPU: NVIDIA GeForce RTX 4060 Laptop GPU
